In [1]:
!pip -q install fastapi uvicorn[standard] pydantic scikit-learn pandas numpy joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.7/517.7 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.8/456.8 kB 9.9 MB/s eta 0:00:00


In [9]:
%%writefile train_model.py
from pathlib import Path
import pandas as pd, numpy as np, json, joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

DATA_PATH = Path("realty_data.csv")
TARGET = "price"
FEATURES = ["lat", "lon", "total_square", "rooms", "floor"]

OUT_DIR = Path("models")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
df = df[FEATURES + [TARGET]].dropna()

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0, random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae:.2f}, R2: {r2:.3f}")

joblib.dump(model, OUT_DIR / "model.pkl")
meta = {
    "feature_names": FEATURES,
    "defaults": X.median(numeric_only=True).to_dict(),
    "target": TARGET,
    "note": "Ridge-регрессия по числовым признакам lat, lon, total_square, rooms, floor"
}
with open(OUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("модель сохранена в models/")

Overwriting train_model.py


In [10]:
!python train_model.py


MAE: 9474943.09, R2: 0.718
модель сохранена в models/


In [11]:
%%writefile app.py
from pathlib import Path
import json, joblib
import pandas as pd
from fastapi import FastAPI, Depends
from pydantic import BaseModel, Field, create_model

MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / "model.pkl"
META_PATH  = MODEL_DIR / "metadata.json"

with open(META_PATH, "r", encoding="utf-8") as f:
    META = json.load(f)

FEATURES = META["feature_names"]
DEFAULTS = META["defaults"]
TARGET   = META["target"]

MODEL = joblib.load(MODEL_PATH)

_query_fields = {f: (float | None, Field(default=DEFAULTS.get(f, None))) for f in FEATURES}
_body_fields  = {f: (float | None, Field(default=None)) for f in FEATURES}

FeaturesQuery = create_model("FeaturesQuery", **_query_fields)
FeaturesBody  = create_model("FeaturesBody", __base__=BaseModel, **_body_fields)

app = FastAPI(title="Realty Price API", description="Прогноз цены недвижимости", version="1.0")

def make_row(d: dict) -> pd.DataFrame:
    return pd.DataFrame([{f: float(d.get(f, DEFAULTS[f])) for f in FEATURES}])

@app.get("/health")
def health():
    return {"status": "ok", "n_features": len(FEATURES), "features": FEATURES}

@app.get("/meta")
def meta():
    return META

@app.get("/predict_get")
def predict_get(q: FeaturesQuery = Depends()):
    X = make_row(q.model_dump())
    y = float(MODEL.predict(X)[0])
    return {"prediction": y, "features": FEATURES}

@app.post("/predict_post")
def predict_post(body: FeaturesBody):
    X = make_row(body.model_dump())
    y = float(MODEL.predict(X)[0])
    return {"prediction": y, "features": FEATURES}

Overwriting app.py


In [12]:
!pip -q install fastapi uvicorn[standard] joblib pydantic pandas scikit-learn requests
import subprocess, time, re, requests, os

!pkill -f "uvicorn app:app" || true
!pkill -f cloudflared || true

server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

# ждём пока сервер поднимется
for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=0.5).status_code == 200:
            break
    except Exception:
        time.sleep(0.5)

if not os.path.exists("cloudflared"):
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

pat = re.compile(r"https?://[0-9a-zA-Z\-\.]+trycloudflare\.com")
url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    if not line: time.sleep(0.1); continue
    m = pat.search(line)
    if m:
        url = m.group(0)
        break

print("🌐 Swagger UI:", url + "/docs" if url else "❌ не удалось получить ссылку")

^C
^C
🌐 Swagger UI: https://pendant-sell-concrete-librarian.trycloudflare.com/docs
